In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List, Tuple, Union

import duckdb
import pandas as pd

true_path = Path("brown_outputs_manually_fixed.json")   # <- set to your "true" file (e.g., brown_outputs_true.json)
false_path = Path("brown_outputs_manually_fixed.json") # <- set to your "false" file (e.g., brown_outputs_false.json)

In [2]:
# ---- parser for "quasi-JSON": repeated JSON objects, not wrapped in an array ----
def parse_quasi_json_objects(text: str) -> List[Dict[str, Any]]:
    """
    Parse a string containing multiple JSON objects back-to-back with arbitrary whitespace/newlines.
    Ignores '...' lines and other non-JSON noise.
    """
    # Remove obvious noise lines like "..." that often appear in snippets/logs
    filtered_lines = text.splitlines()
    cleaned = "\n".join([ln for ln in filtered_lines if ln.strip() != "..."])

    decoder = json.JSONDecoder()
    idx = 0
    n = len(cleaned)
    objs: List[Dict[str, Any]] = []

    while idx < n:
        # Skip whitespace
        while idx < n and cleaned[idx].isspace():
            idx += 1
        if idx >= n:
            break

        # If we're not at a JSON object start, skip forward until we are
        if cleaned[idx] != "{":
            idx += 1
            continue

        obj, end = decoder.raw_decode(cleaned, idx)
        if not isinstance(obj, dict):
            raise ValueError(f"Expected JSON object (dict) at pos {idx}, got {type(obj)}")
        objs.append(obj)
        idx = end

    return objs

In [ ]:
con = duckdb.connect(database=":memory:")
a =  """paper_type.screening::BOOLEAN AS screening,
        paper_type.known::BOOLEAN     AS known,
        screens.fragment::BOOLEAN     AS fragment,
        screens.undirected::BOOLEAN   AS undirected,
        screens.directed::BOOLEAN     AS directed,
        screens.virtual::BOOLEAN      AS virtual,
        screens.del::BOOLEAN          AS del,
        title::VARCHAR                AS title,"""
b =  """CASE WHEN paper_type.screening AND label THEN 'TP'
            WHEN paper_type.screening AND NOT label THEN 'FP'
            WHEN NOT paper_type.screening AND NOT label THEN 'TN'
            WHEN NOT paper_type.screening AND label THEN 'FN' END
        AS result
"""
con.execute(f"""
    CREATE OR REPLACE TABLE examples AS
    SELECT 
        {a}
        True                          AS label,
        {b}
    FROM read_json_auto('{str(true_path)}')
    UNION ALL
    SELECT 
        {a}
        False                         AS label,
        {b}
    FROM read_json_auto('{str(false_path)}');
""")

# Optional sanity checks (keep or remove)
print(con.execute("DESCRIBE examples").fetchdf())
print(con.execute("SELECT label, COUNT(*) AS n FROM examples GROUP BY label ORDER BY label").fetchdf())
display(con.execute("SELECT * FROM examples").df())

  column_name column_type null   key default extra
0   screening     BOOLEAN  YES  None    None  None
1       known     BOOLEAN  YES  None    None  None
2    fragment     BOOLEAN  YES  None    None  None
3  undirected     BOOLEAN  YES  None    None  None
4    directed     BOOLEAN  YES  None    None  None
5     virtual     BOOLEAN  YES  None    None  None
6         del     BOOLEAN  YES  None    None  None
7       title     VARCHAR  YES  None    None  None
8       label     BOOLEAN  YES  None    None  None
9      result     VARCHAR  YES  None    None  None
   label   n
0  False  59
1   True  59


,screening,known,fragment,undirected,directed,virtual,del,title,label,result
0,True,False,False,False,False,False,False,title,True,TP
1,True,False,False,True,False,False,False,"Discovery of N-(4-(2,4-Difluorophenoxy)-3-(6-m...",True,TP
2,True,True,False,True,False,False,False,Optimization of a Series of Bivalent Triazolop...,True,TP
3,True,False,False,False,False,False,False,Identification of a Benzoisoxazoloazepine Inhi...,True,TP
4,True,True,False,True,False,False,False,Discovery of a Novel and Selective Indoleamine...,True,TP
...,...,...,...,...,...,...,...,...,...,...
113,True,False,False,False,False,False,False,Development of a Dual-Acting Antibacterial Age...,False,FP
114,True,False,False,False,False,False,True,Discovery of a First-in-Class Receptor Interac...,False,FP
115,True,False,False,True,False,False,False,Discovery of N-{4-[5-(4-Fluorophenyl)-3-methyl...,False,FP
116,True,False,False,False,False,False,False,"Discovery of Clinical Candidate 1-{[(2S,3S,4S)...",False,FP


In [22]:
# Compute Accuracy, Precision, Recall, Specificity from examples.result ∈ {TP, FP, TN, FN}

row = con.execute("""
SELECT
  SUM(CASE WHEN result = 'TP' THEN 1 ELSE 0 END) AS TP,
  SUM(CASE WHEN result = 'FP' THEN 1 ELSE 0 END) AS FP,
  SUM(CASE WHEN result = 'TN' THEN 1 ELSE 0 END) AS TN,
  SUM(CASE WHEN result = 'FN' THEN 1 ELSE 0 END) AS FN
FROM examples
""").fetchone()

TP, FP, TN, FN = map(int, row)
N = TP + FP + TN + FN

def safe_div(num, den):
    return float('nan') if den == 0 else num / den

accuracy     = safe_div(TP + TN, N)
precision    = safe_div(TP, TP + FP)
recall       = safe_div(TP, TP + FN)
specificity  = safe_div(TN, TN + FP)

print(f"TP={TP} FP={FP} TN={TN} FN={FN} (N={N})")
print(f"Accuracy    : {accuracy:.6f}")
print(f"Precision   : {precision:.6f}")
print(f"Recall      : {recall:.6f}")
print(f"Specificity : {specificity:.6f}")

TP=59 FP=59 TN=0 FN=0 (N=118)
Accuracy    : 0.500000
Precision   : 0.500000
Recall      : 1.000000
Specificity : 0.000000
